# 🚀 FEDHEP POST-MIDTERM EXPERIMENTS — PART 1 OF 2
## 🌐 STUDY A: Peer Clusters ($K$) Sensitivity & 🔬 STUDY D: TTT Removal Ablation

> **Mục tiêu nghiên cứu**:
> 1. **Study A (Idea A của Supervisor)**: Khảo sát sự ảnh hưởng của số lượng peer cluster $K \in \{1, 2, 3, 5, 8, 10, 15\}$ đến độ chính xác cá nhân hóa (personalized accuracy) và độ công bằng (bottom-10% fairness) dưới 3 mức độ non-IID ($lpha \in \{0.05, 0.1, 0.5\}$) với nhiều seeds thống kê.
> 2. **Study D (Hypothesis Verification)**: Kiểm chứng thực nghiệm giả thuyết về sự đóng góp của **Layer 3: Temporal Trust Tracking (TTT)**. So sánh trực tiếp FedHEP khi **bật (ON)** và **tắt hoàn toàn (OFF)** Layer TTT để chứng minh con số ngưỡng tĩnh 0.25 khiến TTT có 0% đóng góp và phân tích các chi phí vận hành (latency, memory overhead) của thuật toán.
>
> ⚡ **Chạy song song**: Notebook này chạy độc lập với **Part 2** (nghiên cứu về Attacks & Majority $q \ge 50\%$) trên Kaggle GPU để tối ưu thời gian chạy.

In [ ]:
import os
import sys

# 1. Repository Setup
ROOT = '/kaggle/working/Topology-aware-FDL'
if not os.path.exists(ROOT):
    print('🚀 Cloning repository from GitHub...')
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
else:
    print('🔄 Repository exists, pulling latest updates from main...')
    !cd {ROOT} && git pull origin main

%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 2. DEDICATED OUTPUT DIRECTORY (Outside of repo, directly visible in Kaggle Output)
if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working/oreh_results_part1'
else:
    OUT_DIR = os.path.abspath('./oreh_results_part1')

SANITY_DIR = os.path.join(OUT_DIR, 'sanity')
EVAL_DIR = os.path.join(OUT_DIR, 'eval')
FIG_DIR = os.path.join(OUT_DIR, 'figures')

os.makedirs(SANITY_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('✅ Working Directory:', os.getcwd())
print('📁 Dedicated Results Directory:', OUT_DIR)
print('📊 CSV Tables Directory:', EVAL_DIR)
print('📈 Figures Directory:', FIG_DIR)
# 3. ROBUST OUTPUT & EMERGENCY PACKAGING (Guarantees output saved even if timeout / limit reached)
import shutil
import atexit
import signal

ZIP_BASE = '/kaggle/working/oreh_part1_summary' if os.path.exists('/kaggle/working') else './oreh_part1_summary'

def package_results_zip():
    try:
        if os.path.exists(OUT_DIR) and os.listdir(OUT_DIR):
            shutil.make_archive(ZIP_BASE, 'zip', OUT_DIR)
            print(f'📦 [Auto-Package] Output archive updated: {ZIP_BASE}.zip')
    except Exception as e:
        print(f'⚠️ Warning during packaging: {e}')

# Register exit handlers for process termination or Kaggle 12h timeout / cancellation signals
atexit.register(package_results_zip)
try:
    signal.signal(signal.SIGTERM, lambda s, f: (package_results_zip(), sys.exit(0)))
    signal.signal(signal.SIGINT, lambda s, f: (package_results_zip(), sys.exit(0)))
except Exception:
    pass


In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy

In [ ]:
from src.data.dataset import _auto_link_dataset
data_dir = _auto_link_dataset('cifar100')
print('📂 CIFAR-100 Dataset linked at:', data_dir)

In [ ]:
# ==============================================================================
# Cell 4: 1-Round Pre-Flight Sanity Check
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running 1-round pre-flight sanity check on CIFAR-100 using {device}...')

sanity_runner = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=1,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=1000,
    test_subset=500,
    data_dir=data_dir,
    output_dir=SANITY_DIR,
)

for k_test in [2, 3]:
    test_res = sanity_runner.run_single_experiment(
        defense_scope='both',
        num_clusters=k_test,
        attack_type='none',
        byzantine_rate=0.0,
        seed=42,
    )
    print(f'✅ Sanity Check K={k_test} PASSED! Acc: {test_res["final_test_acc"]}% | Latency: {test_res["total_defense_latency_ms"]} ms')

print('\n🚀 Pre-Flight Sanity Checks Successfully Completed! Ready for Study A & D sweeps.')

## 🌐 STUDY A (Idea A): Multi-Seed Peer Clusters ($K$) Sensitivity Sweep
Khảo sát số lượng peer cluster $K \in \{1, 2, 3, 5, 8, 10, 15\}$ trên 3 mức độ non-IID:
- **Extreme Skew**: $lpha = 0.05$ (mỗi client chỉ nắm 1-2 classes chủ đạo)
- **Severe Skew**: $lpha = 0.1$ (độ lệch nhãn cao)
- **Moderate Skew**: $lpha = 0.5$ (chuẩn benchmark phổ biến)

* **Seeds**: $\{42, 123, 7\}$ (cho phép vẽ standard deviation error bands)
* **Output File**: `/kaggle/working/oreh_results_part1/eval/cifar100_postmid_k_clusters.csv` (ghi realtime sau mỗi lượt chạy)

In [ ]:
# ==============================================================================
# Cell 6: [STUDY A] Multi-Seed Peer Clusters (K) Sensitivity Sweep
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_a = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
)

# Chạy sweep đa seeds: K in [1, 2, 3, 5, 8, 10, 15] across alphas [0.05, 0.1, 0.5]
df_k = runner_a.run_postmid_k_clusters(
    k_values=[1, 2, 3, 5, 8, 10, 15],
    alphas=[0.05, 0.1, 0.5],
    seeds=[42, 123, 7],
    byzantine_rate=0.0,
)

print('\n--- STUDY A (K CLUSTERS SWEEP) SUMMARY TABLE ---')
cols_k = ['num_clusters', 'alpha', 'seed', 'final_test_acc', 'bottom_10_fairness', 'total_defense_latency_ms']
if not df_k.empty:
    print(df_k[[c for c in cols_k if c in df_k.columns]].to_string(index=False))
# Save progress and update zip archive immediately
package_results_zip()


## 🔬 STUDY D (Hypothesis Verification): TTT Layer Removal / Ablation (ON vs OFF)
Thực nghiệm kiểm chứng giả thuyết: **Cơ chế Temporal Trust Tracking (TTT) trong Defended FedHEP có thực sự đóng góp vào độ chính xác hay không?**
- **Cơ chế hoạt động**: FedHEP nguyên bản đặt ngưỡng cách ly tĩnh $0.25$. Do trọng số soft cosine ở Lớp 2 không bao giờ khiến điểm uy tín EMA tụt dưới $0.25$, không một attacker nào bị loại bỏ.
- **Thực nghiệm đối chứng**: So sánh trực tiếp FedHEP với `enable_ttt = True` (mặc định) và `enable_ttt = False` (bỏ hoàn toàn Layer 3 TTT) dưới 2 dạng tấn công:
  - `sign_flip` (tấn công đối xứng dấu)
  - `alie` (A Little Is Enough — tấn công vi điều chỉnh trong dải phương sai)
  - Trên các mức attacker fraction: $q \in \{0.2, 0.3\}$
  - Lặp qua 2 seeds $\{42, 123\}$

* **Output File**: `/kaggle/working/oreh_results_part1/eval/cifar100_postmid_ttt_ablation.csv` (ghi realtime sau mỗi lượt chạy)

In [ ]:
# ==============================================================================
# Cell 8: [STUDY D] TTT Layer Removal / Ablation Study (ON vs OFF)
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_d = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
)

df_ttt = runner_d.run_postmid_ttt_ablation(
    attacks=['sign_flip', 'alie'],
    byzantine_rates=[0.2, 0.3],
    ttt_options=[True, False],
    seeds=[42, 123],
    defense_scope='both',
)

print('\n--- STUDY D (TTT ABLATION: ON vs OFF) SUMMARY TABLE ---')
cols_ttt = ['attack_type', 'byzantine_rate', 'enable_ttt', 'seed', 'final_test_acc', 'byzantine_trust_score', 'total_defense_latency_ms']
if not df_ttt.empty:
    print(df_ttt[[c for c in cols_ttt if c in df_ttt.columns]].to_string(index=False))
# Save progress and update zip archive immediately
package_results_zip()


## 📊 Publication-Quality Visualizations & Statistical Analysis
Vẽ biểu đồ phân tích cho cả 2 nghiên cứu:
1. **Study A**: Đường cong tương quan số cụm $K$ vs Accuracy kèm shaded error bands ($\pm 1 	ext{ std}$) và Bottom-10% Fairness.
2. **Study D**: Biểu đồ Bar đối chứng trực tiếp giữa **TTT ON** vs **TTT OFF**, minh họa trực quan sự tương đương (zero difference) khẳng định giả thuyết TTT không đóng góp.

In [ ]:
# ==============================================================================
# Cell 10: Plotting Study A & Study D Results
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------------------
# 1. Plot Study A: Accuracy & Fairness vs K
# ------------------------------------------------------------------------------
k_file = os.path.join(EVAL_DIR, 'cifar100_postmid_k_clusters.csv')
if os.path.exists(k_file):
    df_k = pd.read_csv(k_file)
    if not df_k.empty:
        plt.figure(figsize=(9, 5.5))
        summary_k = df_k.groupby(['alpha', 'num_clusters'])['final_test_acc'].agg(['mean', 'std']).reset_index()
        colors = {'0.05': '#d62728', '0.1': '#ff7f0e', '0.5': '#1f77b4'}
        for a in sorted(summary_k['alpha'].unique()):
            sub = summary_k[summary_k['alpha'] == a].sort_values('num_clusters')
            k_vals = sub['num_clusters'].values
            means = sub['mean'].values
            stds = sub['std'].fillna(0).values
            c = colors.get(str(a), None)
            plt.plot(k_vals, means, marker='s', linewidth=2.5, label=f'Dirichlet $\\alpha = {a}$', color=c)
            plt.fill_between(k_vals, means - stds, means + stds, alpha=0.18, color=c)
            
        plt.title('CIFAR-100: Impact of Number of Peer Clusters ($K$) on Accuracy across Skews', fontsize=12, fontweight='bold')
        plt.xlabel('Number of Peer Clusters ($K$)', fontsize=11)
        plt.ylabel('Personalized Test Accuracy (%)', fontsize=11)
        plt.xticks([1, 2, 3, 5, 8, 10, 15])
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend(frameon=True, fontsize=10)
        plt.tight_layout()
        acc_fig = os.path.join(FIG_DIR, 'postmid_a_k_cluster_multiseed_curve.png')
        plt.savefig(acc_fig, dpi=200)
        plt.show()
        print(f'✅ Saved accuracy plot to: {acc_fig}')

# ------------------------------------------------------------------------------
# 2. Plot Study D: TTT Ablation (ON vs OFF) Verification Bar Chart
# ------------------------------------------------------------------------------
ttt_file = os.path.join(EVAL_DIR, 'cifar100_postmid_ttt_ablation.csv')
if os.path.exists(ttt_file):
    df_ttt = pd.read_csv(ttt_file)
    if not df_ttt.empty:
        plt.figure(figsize=(10, 5.5))
        summary_ttt = df_ttt.groupby(['attack_type', 'byzantine_rate', 'enable_ttt'])['final_test_acc'].mean().reset_index()
        
        # Build grouped comparison
        configs = summary_ttt.groupby(['attack_type', 'byzantine_rate']).groups.keys()
        labels = [f'{atk.upper()}\n(q={q})' for atk, q in configs]
        x = np.arange(len(labels))
        width = 0.35

        on_accs = []
        off_accs = []
        for atk, q in configs:
            sub = summary_ttt[(summary_ttt['attack_type'] == atk) & (summary_ttt['byzantine_rate'] == q)]
            val_on = sub[sub['enable_ttt'] == True]['final_test_acc'].values
            val_off = sub[sub['enable_ttt'] == False]['final_test_acc'].values
            on_accs.append(val_on[0] if len(val_on) > 0 else 0.0)
            off_accs.append(val_off[0] if len(val_off) > 0 else 0.0)

        plt.bar(x - width/2, on_accs, width, label='TTT Active (ON)', color='#1f77b4', alpha=0.85)
        plt.bar(x + width/2, off_accs, width, label='TTT Removed (OFF)', color='#ff7f0e', alpha=0.85)

        plt.title('CIFAR-100: TTT Layer Ablation (Zero Impact on Robustness/Accuracy)', fontsize=12, fontweight='bold')
        plt.xlabel('Attack Configuration', fontsize=11)
        plt.ylabel('Personalized Test Accuracy (%)', fontsize=11)
        plt.xticks(x, labels, fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.legend(frameon=True, fontsize=10)
        plt.tight_layout()
        ttt_fig = os.path.join(FIG_DIR, 'postmid_d_ttt_ablation.png')
        plt.savefig(ttt_fig, dpi=200)
        plt.show()
        print(f'✅ Saved TTT ablation plot to: {ttt_fig}')

## 📦 1-Click ZIP Packaging for Direct Download
Tự động nén toàn bộ thư mục `/kaggle/working/oreh_results_part1/` thành 1 file ZIP duy nhất. Bạn có thể download trực tiếp từ tab Output bên phải màn hình Kaggle mà không cần tìm kiếm trong repo code!

In [ ]:
# ==============================================================================
# Final Packaging for Direct Download
# ==============================================================================
package_results_zip()

print('\n🎉 ALL EXPERIMENTS & PACKAGING COMPLETED!')
print(f'📁 Dedicated Folder: {OUT_DIR}')
print(f'📦 Direct Download File: {ZIP_BASE}.zip')
print('You can download the summary zip file directly from the Kaggle Output sidebar!')
